# Stacking (P3_SNP, P3_LA, P3_SNP-LA)

Inputs: the out-of-fold predictions written by `01_snp_pipelines.ipynb` and `02_local_ancestry_pipelines.ipynb`, and
`y` (one-column DataFrame of 0/1 labels indexed by sample ID).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, balanced_accuracy_score, precision_score, recall_score
from imblearn.metrics import specificity_score
import scipy.stats as stats

os.makedirs('results', exist_ok=True)

In [ ]:
all_preds_data = []
all_y_test =[]

for i in range(10):
    a = pd.DataFrame(pd.read_csv(f'pred_probs/snps_lasso_split{i}.csv', header=None, index_col=0).iloc[:, 0])
    b = pd.DataFrame(pd.read_csv(f'pred_probs/snps_rfe_rf_split{i}.csv', header=None, index_col=0).iloc[:, 1])
    c = pd.DataFrame(pd.read_csv(f'pred_probs/ancs_lasso_split{i}.csv', header=None, index_col=0).iloc[:, 0])
    d = pd.DataFrame(pd.read_csv(f'pred_probs/ancs_en_rf_split{i}.csv', header=None, index_col=0).iloc[:, 1])
    a.columns, b.columns, c.columns, d.columns = ['P1_SNP'], ['P2_SNP'], ['P1_LA'], ['P2_LA']

    e = a.merge(b, left_index=True, right_index=True).merge(c, left_index=True, right_index=True).merge(d, left_index=True, right_index=True)
    all_preds_data.append(e)

    y_temp = e.merge(y, left_index=True, right_index=True).iloc[:, -1]
    all_y_test.append(y_temp)

In [ ]:
def pred_cv(all_preds_data, all_y_test, flex=3):

    y_true =[]
    y_pred = []
    y_probs = []

    for i in range(10):

        X_test = all_preds_data[i]
        y_test = all_y_test[i]

        p3_snp = 0.5 * X_test['P1_SNP'] + 0.5 * X_test['P2_SNP']
        p3_la = 0.5 * X_test['P1_LA'] + 0.5 * X_test['P2_LA']

        if flex==3:
            y_hat_probs = 0.68 * p3_snp + 0.32 * p3_la
            y_hat = [int(g) for g in np.round(y_hat_probs, decimals=4) >= 0.52]

        elif flex==4:
            y_hat_probs = p3_snp
            y_hat = [int(g) for g in np.round(y_hat_probs, decimals=4) >= 0.50]

        elif flex==5:
            y_hat_probs = p3_la
            y_hat = [int(g) for g in np.round(y_hat_probs, decimals=4) >= 0.50]

        y_true.append(y_test)
        y_pred.append(y_hat)
        y_probs.append(y_hat_probs)

    return y_true, y_pred, y_probs

## P3_SNP

In [ ]:
y_true, y_pred, y_probs = pred_cv(all_preds_data, all_y_test, flex=4)

In [ ]:
roc_auc = []
pr_auc = []
precs = []
recs = []
specifs = []
balanced_accs =[]
tprs = []

mean_fpr = np.linspace(0, 1, 10)

for i, j, k in zip(y_true, y_pred, y_probs):
    roc_auc.append(roc_auc_score(i, k, average='weighted'))
    precs.append(precision_score(i, j, average='weighted'))
    recs.append(recall_score(i, j, average='weighted'))
    precision_pr, recall_pr, _ = precision_recall_curve(i, k)
    specifs.append(specificity_score(i, j, average='weighted'))
    
    balanced_accs.append(balanced_accuracy_score(i, j))

    fpr, tpr, _ = roc_curve(i, k)
    tprs.append(np.interp(mean_fpr, fpr, tpr))

In [ ]:
print(roc_auc)
print("roc_auc: ", np.mean(roc_auc), np.std(roc_auc))

print(precs)
print("precision_score: ", np.mean(precs), np.std(precs))

print(recs)
print("recall_score: ", np.mean(recs), np.std(recs))

print(specifs)
print("specificity: ", np.mean(specifs), np.std(specifs))

print(balanced_accs)
print("balanced_auc: ", np.mean(balanced_accs), np.std(balanced_accs))

In [ ]:
temp__ = pd.DataFrame({
    "roc_auc": np.round(roc_auc, decimals=4),
    "precision_score" : np.round(precs, decimals=4) ,
    "recall_score" : np.round(recs, decimals=4),
    "specificity": np.round(specifs, decimals=4),
    "balanced_auc": np.round(balanced_accs, decimals=4),
})

temp__.to_csv('results/p3_snp.csv')

## P3_LA

In [ ]:
y_true, y_pred, y_probs = pred_cv(all_preds_data, all_y_test, flex=5)

In [ ]:
roc_auc = []
pr_auc = []
precs = []
recs = []
specifs = []
balanced_accs =[]
tprs = []

mean_fpr = np.linspace(0, 1, 10)

for i, j, k in zip(y_true, y_pred, y_probs):
    roc_auc.append(roc_auc_score(i, k, average='weighted'))
    precs.append(precision_score(i, j, average='weighted'))
    recs.append(recall_score(i, j, average='weighted'))
    precision_pr, recall_pr, _ = precision_recall_curve(i, k)
    specifs.append(specificity_score(i, j, average='weighted'))
    
    balanced_accs.append(balanced_accuracy_score(i, j))

    fpr, tpr, _ = roc_curve(i, k)
    tprs.append(np.interp(mean_fpr, fpr, tpr))

In [ ]:
print(roc_auc)
print("roc_auc: ", np.mean(roc_auc), np.std(roc_auc))

print(precs)
print("precision_score: ", np.mean(precs), np.std(precs))

print(recs)
print("recall_score: ", np.mean(recs), np.std(recs))

print(specifs)
print("specificity: ", np.mean(specifs), np.std(specifs))

print(balanced_accs)
print("balanced_auc: ", np.mean(balanced_accs), np.std(balanced_accs))

In [ ]:
temp__ = pd.DataFrame({
    "roc_auc": np.round(roc_auc, decimals=4),
    "precision_score" : np.round(precs, decimals=4) ,
    "recall_score" : np.round(recs, decimals=4),
    "specificity": np.round(specifs, decimals=4),
    "balanced_auc": np.round(balanced_accs, decimals=4),
})

temp__.to_csv('results/p3_la.csv')

## P3_SNP-LA

In [ ]:
y_true, y_pred, y_probs = pred_cv(all_preds_data, all_y_test, flex=3)

In [ ]:
roc_auc = []
pr_auc = []
precs = []
recs = []
specifs = []
balanced_accs =[]
tprs = []

mean_fpr = np.linspace(0, 1, 10)

for i, j, k in zip(y_true, y_pred, y_probs):
    roc_auc.append(roc_auc_score(i, k, average='weighted'))
    precs.append(precision_score(i, j, average='weighted'))
    recs.append(recall_score(i, j, average='weighted'))
    precision_pr, recall_pr, _ = precision_recall_curve(i, k)
    specifs.append(specificity_score(i, j, average='weighted'))
    
    balanced_accs.append(balanced_accuracy_score(i, j))

    fpr, tpr, _ = roc_curve(i, k)
    tprs.append(np.interp(mean_fpr, fpr, tpr))

In [ ]:
print(roc_auc)
print("roc_auc: ", np.mean(roc_auc), np.std(roc_auc))

print(precs)
print("precision_score: ", np.mean(precs), np.std(precs))

print(recs)
print("recall_score: ", np.mean(recs), np.std(recs))

print(specifs)
print("specificity: ", np.mean(specifs), np.std(specifs))

print(balanced_accs)
print("balanced_auc: ", np.mean(balanced_accs), np.std(balanced_accs))

In [ ]:
temp__ = pd.DataFrame({
    "roc_auc": np.round(roc_auc, decimals=4),
    "precision_score" : np.round(precs, decimals=4) ,
    "recall_score" : np.round(recs, decimals=4),
    "specificity": np.round(specifs, decimals=4),
    "balanced_auc": np.round(balanced_accs, decimals=4),
})

temp__.to_csv('results/p3_snp_la.csv')

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(15, 6))

axes = axes.flatten()

for i, ax in enumerate(axes):
    temp = np.asarray(y_probs[i])
    labels_i = np.asarray(y_true[i]).ravel()

    positive = temp[labels_i == 1]
    negative = temp[labels_i == 0]

    ax.hist(positive, color='red', label='Cases')
    ax.hist(negative, color='blue', label='Controls')
    ax.set_title(f'Fold {i+1} (n={len(y_true[i])})')

fig.supxlabel('Prediction Probabilities')
fig.supylabel('Number of Individuals')
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right')

plt.tight_layout()
    
plt.show()

## Paired t-test: P3_SNP-LA vs P3_SNP

In [ ]:
main = pd.read_csv('results/p3_snp.csv').iloc[:, 1:]
temp__ = pd.read_csv('results/p3_snp_la.csv').iloc[:, 1:]
metric_names = ["roc_auc","precision_score","recall_score", "specificity", "balanced_auc"]

for i in range(len(main.columns)):
    t_stats, pvals = stats.ttest_rel(main.iloc[:, i], temp__.iloc[:, i])
    print("{}: \nt_stats: {}\tpvals: {}".format(metric_names[i], t_stats, pvals))